## Brain Map Visualization - Cortical Volume

### Overview

This notebook visualizes parcelwise **effect sizes (Hedges' g)** of cortical volume
differences between groups, projected onto the **Desikan-Killiany (dk)** atlas using the
`ggseg` R package (Mowinckel & Vidal-Piñeiro, 2020).

**Effect size metric:** Hedges' g - a bias-corrected standardized mean difference.
Positive values = first group > second group; negative = first < second.

**Atlas:** Desikan-Killiany (`dk`), 34 regions per hemisphere = 68 parcels total.

**Covariate note:** Cortical volume was residualized on **age, SEX, and eTIV** (estimated
total intracranial volume) prior to group comparisons. eTIV captures head/brain size
differences that scale volumetric measures.

**Contrasts:** De Novo PD vs HC · Prodromal PD vs HC · De Novo PD vs Prodromal PD

**Pipeline overview:**
1. Load aligned subject-level data (`df1_id_volume_cortical_aligned.csv`)
2. Compute Welch's t-test + Hedges' g per parcel per contrast
3. Apply FDR correction within each contrast
4. Plot full Hedges' g map and FDR-masked map per contrast
5. Save figures

**Reference:** Mowinckel & Vidal-Piñeiro (2020). Visualization of Brain Statistics With R
Packages ggseg and ggseg3d. *Advances in Methods and Practices in Psychological Science*.

In [ ]:
# ── Install packages if not already available ─────────────────────────────────
# Uncomment on first run:
# install.packages(c("tidyverse", "patchwork", "remotes"))
# remotes::install_github("LCBC-UiO/ggseg")

suppressPackageStartupMessages({
  library(ggseg)
  library(tidyverse)
  library(patchwork)
})

cat("ggseg version:", as.character(packageVersion("ggseg")), "\n")
cat("ggplot2 version:", as.character(packageVersion("ggplot2")), "\n")

### 1. Load pre-computed parcelwise OLS T-statistics

Three CSV files from the main analysis (`volume_cortical_shi.ipynb`), one per contrast.
Each file contains 68 rows (34 LH + 34 RH parcels) with columns:
- `parcel` — NiSpace parcel ID, e.g. `hemi-L_lab-bankssts`
- `Tvalue` — OLS T-statistic (sign convention: T > 0 means reference > first-named group)
- `pvalue` — two-tailed p-value
- `df` — residual degrees of freedom
- `hemi` — hemisphere code (`L` / `R`)

The OLS model included **age, sex, and eTIV** as covariates.

**Why use pre-computed T-stats instead of raw data?**
Cortical volumes scale with total intracranial volume (eTIV). Without eTIV correction,
PD subjects (who tend to have larger TIV in PPMI) appear to have larger raw volumes,
which can inflate effect sizes and bias parcel-level comparisons. The OLS T-statistics
from the main analysis have eTIV regressed out, giving the correct covariate-adjusted result.

In [ ]:
RESULTS_DIR <- "../../results"
FIG_DIR     <- file.path(RESULTS_DIR, "figures", "ggseg_volume_cortical")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)

GROUP_N <- list(
  "De Novo PD vs HC"           = c(n1 = 558, n2 = 192),
  "Prodromal PD vs HC"         = c(n1 = 279, n2 = 192),
  "De Novo PD vs Prodromal PD" = c(n1 = 558, n2 = 279)
)

TTEST_FILES <- list(
  "De Novo PD vs HC"           = file.path(RESULTS_DIR, "de novo pd vs hc_parcelwise_ttest_volume_cortical.csv"),
  "Prodromal PD vs HC"         = file.path(RESULTS_DIR, "prodromal pd vs hc_parcelwise_ttest_volume_cortical.csv"),
  "De Novo PD vs Prodromal PD" = file.path(RESULTS_DIR, "de novo pd vs prodromal pd_parcelwise_ttest_volume_cortical.csv")
)

df_cortical <- imap_dfr(TTEST_FILES, function(path, cname) {
  read_csv(path, show_col_types = FALSE) %>% mutate(contrast = cname)
}) %>%
  filter(!is.na(pvalue))

cat("Loaded:", nrow(df_cortical), "parcel-contrast rows\n")
cat("Contrasts:", paste(unique(df_cortical$contrast), collapse = " | "), "\n")

### 2. Derive Hedges' g from OLS T-statistics

For each of the 68 parcels and 3 contrasts we convert the OLS T-statistic to Hedges' g:

$$g = -T \cdot \sqrt{\frac{1}{n_1} + \frac{1}{n_2}} \cdot J(df)$$

$$J(df) = 1 - \frac{3}{4 \cdot df - 1}$$

**Sign convention:** OLS codes the group variable as `{first-named: 0, reference: 1}`,
so a positive T means reference > first-named. Negating T gives **g > 0 = first-named group
has greater volume** (e.g. for "De Novo PD vs HC", g > 0 means PD larger than HC).

**Why OLS T-stats instead of Welch's t-test on raw volumes?**
Cortical volumes scale with eTIV (total intracranial volume). In PPMI, PD groups tend to have
larger TIV than HC, so raw (unadjusted) volume comparisons can be biased or even reversed in
direction relative to the eTIV-corrected result. Deriving g from the OLS T-statistics (which
include eTIV as a covariate) ensures the effect sizes are properly adjusted.

In [ ]:
# Named list kept for downstream visualization cells that loop over names(contrasts)
contrasts <- setNames(as.list(names(GROUP_N)), names(GROUP_N))

hedges_g_from_t <- function(t, n1, n2, df_resid) {
  # Sign negation: OLS T encodes (reference - first-named), so -T gives first-named - reference.
  # g > 0 means first-named group (e.g. De Novo PD) has greater volume.
  -t * sqrt(1/n1 + 1/n2) * (1 - 3 / (4 * df_resid - 1))
}

df_stats <- df_cortical %>%
  rowwise() %>%
  mutate(g = hedges_g_from_t(
    Tvalue,
    GROUP_N[[contrast]][["n1"]],
    GROUP_N[[contrast]][["n2"]],
    df
  )) %>%
  ungroup() %>%
  mutate(
    hemi   = if_else(hemi == "L", "left", "right"),
    region = str_replace(parcel, ".*_lab-", ""),
    label  = paste0(if_else(hemi == "left", "lh", "rh"), "_", region)
  )

cat("Hedges' g range:", round(range(df_stats$g, na.rm = TRUE), 3), "\n")

### 3. FDR correction

Benjamini-Hochberg FDR applied within each contrast across all 68 parcels.

In [ ]:
df_stats <- df_stats %>%
  group_by(contrast) %>%
  mutate(p_fdr = p.adjust(pvalue, method = "fdr")) %>%
  ungroup()

df_stats %>%
  group_by(contrast) %>%
  summarise(
    n_sig_fdr   = sum(p_fdr < 0.05, na.rm = TRUE),
    n_sig_uncor = sum(pvalue < 0.05, na.rm = TRUE),
    n_total     = n(),
    g_max_abs   = round(max(abs(g), na.rm = TRUE), 3)
  )

### 4. Colour scale

Diverging scale centred at 0; blue = smaller volume, red = larger volume.
Limits ±0.5; adjust `G_LIMIT` if effects are consistently larger.

In [ ]:
G_LIMIT <- 0.5

scale_g <- scale_fill_gradient2(
  low      = "#2166AC",
  mid      = "white",
  high     = "#D6604D",
  midpoint = 0,
  limits   = c(-G_LIMIT, G_LIMIT),
  oob      = scales::squish,
  name     = "Hedges' g",
  na.value = "grey85"
)

### 5. Brain maps per contrast

Left panel: full Hedges' g map. Right panel: FDR-masked (q < 0.05).

Following ggseg 2.x API: only `label` + fill variable passed to `ggplot()`.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5)

# Build 4-view atlas once (lateral + medial only)
if (!exists("dk_4view")) {
  dk_4view <- dk()
  dk_4view$data[[1]] <- dk_4view$data[[1]] %>% filter(view %in% c("lateral", "medial"))
}

for (cname in names(contrasts)) {

  p_full <- ggplot(df_stats %>% filter(contrast == cname) %>% select(label, g)) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = g), colour = "white",
               position = position_brain("horizontal")) +
    scale_g +
    labs(title = cname, subtitle = "Cortical volume - Hedges' g (all parcels; eTIV-corrected)") +
    theme_brain2() +
    theme(plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  p_masked <- ggplot(
    df_stats %>% filter(contrast == cname) %>%
      mutate(g_sig = if_else(p_fdr < 0.05, g, NA_real_)) %>% select(label, g_sig)
  ) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = g_sig), colour = "white",
               position = position_brain("horizontal")) +
    scale_g +
    labs(title = cname, subtitle = "Cortical volume - FDR-masked (q < 0.05; eTIV-corrected)") +
    theme_brain2() +
    theme(plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  combined <- p_full + p_masked + plot_layout(guides = "collect") & theme(legend.position = "right")
  print(combined)
  fname <- tolower(str_replace_all(cname, " ", "_"))
  ggsave(file.path(FIG_DIR, paste0(fname, ".png")), combined, width = 14, height = 5, dpi = 300)
  cat("Saved:", fname, "\n")
}

### 5b. Paper-style brain maps (sequential red scale, FDR-only)

Replicating the style of Laansma et al. (following the figure caption):
> *"Cohen's d values were calculated and are presented in the figure when the FDR-adjusted
> p value reached < 0.05. Darker red indicates more atrophy."*

**Design choices:**
- **Sequential white → dark red** scale: encodes *magnitude* of atrophy, not direction
- **Absolute |Hedges' g|** displayed — only for FDR-significant parcels
- **Non-significant parcels** → very light grey (`#f5f5f5`) so the brain outline stays visible
- **Grey parcel borders** (`grey60`) make individual parcels distinguishable
- One panel per contrast (not paired full + masked)

In [ ]:
FIG_DIR_PAPER <- file.path(FIG_DIR, "paper_style")
dir.create(FIG_DIR_PAPER, recursive = TRUE, showWarnings = FALSE)

G_LIMIT_PAPER <- ceiling(max(abs(df_stats$g), na.rm = TRUE) * 10) / 10
scale_g_paper <- scale_fill_gradient(
  low = "white", high = "#B2182B",
  limits = c(0, G_LIMIT_PAPER), oob = scales::squish,
  name = "|Hedges' g|", na.value = "#f5f5f5"
)

if (!exists("dk_4view")) {
  dk_4view <- dk()
  dk_4view$data[[1]] <- dk_4view$data[[1]] %>% filter(view %in% c("lateral", "medial"))
}

cat("Paper scale upper limit:", G_LIMIT_PAPER, "\n")
options(repr.plot.width = 10, repr.plot.height = 4)

for (cname in names(contrasts)) {
  plot_data <- df_stats %>%
    filter(contrast == cname) %>%
    mutate(g_abs = if_else(p_fdr < 0.05, abs(g), NA_real_)) %>%
    select(label, g_abs)

  p_paper <- ggplot(plot_data) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = g_abs), colour = "grey60",
               position = position_brain("horizontal")) +
    scale_g_paper +
    labs(title = cname, subtitle = "Cortical volume - |Hedges' g| (FDR q < 0.05 only; eTIV-corrected)") +
    theme_brain2() +
    theme(plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  print(p_paper)
  fname <- paste0("paper_", tolower(str_replace_all(cname, " ", "_")))
  ggsave(file.path(FIG_DIR_PAPER, paste0(fname, ".png")), p_paper, width = 10, height = 4, dpi = 300)
  cat("Saved:", fname, "\n")
}

### 5c. Cortical volume reduction maps: 4-view layout, white → pink

Two publication-ready combined figures (3 contrasts × 4 cortical views each):

**Figure 5c-1 — Unthresholded volume reduction:** All parcels where g < 0 (first-named group has
less cortical volume than the reference group after eTIV correction). Pink intensity = |Hedges' g|.
Grey parcels = no volume loss or larger volume in first-named group. Panel labels: **a / b / c**.

**Figure 5c-2 — FDR-corrected volume reduction:** Only parcels with g < 0 **and** FDR q < 0.05.
No cortical volume parcel survives FDR correction in any contrast. Panel labels: **a / b / c**.

Views shown: LH lateral | LH medial | RH medial | RH lateral (inferior/superior views removed).

In [ ]:
FIG_DIR_ATROPHY <- file.path(FIG_DIR, "atrophy_style")
dir.create(FIG_DIR_ATROPHY, recursive = TRUE, showWarnings = FALSE)

if (!exists("dk_4view")) {
  dk_4view <- dk()
  dk_4view$data[[1]] <- dk_4view$data[[1]] %>% filter(view %in% c("lateral", "medial"))
}

max_atrophy_vol <- ceiling(max(abs(df_stats$g[df_stats$g < 0]), na.rm = TRUE) * 20) / 20
cat("Atrophy scale upper limit:", max_atrophy_vol, "\n")

scale_atrophy_vol <- scale_fill_gradient(
  low = "white", high = "pink",
  limits = c(0, max_atrophy_vol), oob = scales::squish,
  na.value = "grey90", name = "Atrophy\n(|Hedges' g|)"
)

contrast_desc_vol <- c(
  "De Novo PD vs HC"           = "De Novo PD shows less cortical volume than HC",
  "Prodromal PD vs HC"         = "Prodromal PD shows less cortical volume than HC",
  "De Novo PD vs Prodromal PD" = "De Novo PD shows less cortical volume than Prodromal PD"
)

make_vol_panel <- function(cname, fdr_only = FALSE) {
  d <- df_stats %>% filter(contrast == cname)
  n_sig <- sum(d$p_fdr < 0.05 & d$g < 0, na.rm = TRUE)
  if (fdr_only) {
    plot_data <- d %>% mutate(atrophy = if_else(g < 0 & p_fdr < 0.05, abs(g), NA_real_)) %>% select(label, atrophy)
    subt <- if (n_sig == 0) "No parcel survives FDR correction (q < 0.05)" else
      paste0("FDR-significant volume loss: n = ", n_sig, " parcels (q < 0.05)")
  } else {
    plot_data <- d %>% mutate(atrophy = if_else(g < 0, abs(g), NA_real_)) %>% select(label, atrophy)
    n_unc <- sum(d$pvalue < 0.05 & d$g < 0, na.rm = TRUE)
    subt <- paste0(contrast_desc_vol[[cname]], " (pink); grey = no loss  |  ", n_unc, " parcels p < 0.05 uncorrected")
  }
  ggplot(plot_data) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = atrophy), colour = "grey70",
               position = position_brain("horizontal")) +
    scale_atrophy_vol +
    labs(title = cname, subtitle = subt) +
    theme_void() +
    theme(
      plot.title    = element_text(hjust = 0.5, size = 11, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 8.5, colour = "grey35"),
      legend.position = "right"
    )
}

# ── Figure 5c-1: Unthresholded volume reduction (a / b / c) ───────────────
options(repr.plot.width = 10, repr.plot.height = 12)

fig_unc <- wrap_plots(map(names(contrasts), ~ make_vol_panel(.x, FALSE)), ncol = 1) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "Cortical Volume: Regional Volume Reduction Across the Disease Continuum",
    subtitle = paste0(
      "Pink = parcels where the first-named group has less cortical volume (|Hedges' g|, eTIV-corrected, unthresholded).\n",
      "Grey = no volume loss or larger volume relative to reference group.\n",
      "Views: LH lateral | LH medial | RH medial | RH lateral. No parcel survives FDR correction."
    ),
    tag_levels = "a",
    theme = theme(plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
                  plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40"))
  ) & theme(legend.position = "right")

print(fig_unc)
ggsave(file.path(FIG_DIR_ATROPHY, "figure5c1_cortical_volume_unthresholded.png"),
       fig_unc, width = 10, height = 12, dpi = 300)
cat("Saved: figure5c1_cortical_volume_unthresholded.png\n")

# ── Figure 5c-2: FDR-corrected volume reduction (a / b / c) ───────────────
fig_fdr <- wrap_plots(map(names(contrasts), ~ make_vol_panel(.x, TRUE)), ncol = 1) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "Cortical Volume: FDR-corrected Volume Reduction (q < 0.05)",
    subtitle = paste0(
      "Only parcels with statistically significant volume loss (g < 0 and FDR q < 0.05) are coloured.\n",
      "No cortical volume parcel survives FDR correction in any contrast — all panels appear grey."
    ),
    tag_levels = "a",
    theme = theme(plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
                  plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40"))
  ) & theme(legend.position = "right")

print(fig_fdr)
ggsave(file.path(FIG_DIR_ATROPHY, "figure5c2_cortical_volume_fdr_corrected.png"),
       fig_fdr, width = 10, height = 12, dpi = 300)
cat("Saved: figure5c2_cortical_volume_fdr_corrected.png\n")

### 6. Results table

In [ ]:
df_stats %>%
  select(contrast, hemi, region, g, Tvalue, pvalue, p_fdr) %>%
  mutate(
    across(where(is.numeric), \(x) round(x, 4)),
    sig = if_else(p_fdr < 0.05, "*", "")
  ) %>%
  arrange(contrast, desc(abs(g)))

---
## Notes

### Hedges' g vs T-statistic
T is inflated by sample size. For "De Novo PD vs HC" (n≈558 vs n≈192), T-values will
be systematically larger than for "Prodromal PD vs HC" (n≈279 vs n≈192) even if
the underlying effects are similar. Hedges' g removes this bias.

### eTIV correction
Cortical volumes scale with overall brain size. eTIV was included as nuisance covariate
alongside age and SEX in the OLS model. **Running Welch's t-test on raw (unadjusted)
volumes without eTIV can produce inflated or even reversed effect sizes** — this is why
this notebook loads pre-computed OLS T-statistics rather than the raw aligned data file.

### Data source
Effect sizes are derived from OLS T-statistics saved in
`results/*_parcelwise_ttest_volume_cortical.csv` by `volume_cortical_shi.ipynb`.

### ggseg 2.x API
- Atlases: `dk()`, `aseg()` (functions, not data objects)
- Join key: `label` column only (`"lh_bankssts"`, …)
- Pass only `label` + fill variable to `ggplot()`
- Use `position_brain("horizontal")` ("stacked" broken in v2.1.0)

**Group codes:** CONCOHORT 1 = De Novo PD · 2 = HC · 4 = Prodromal PD

### 5e. Diverging pastel maps: both directions (light pink / light blue)

**Appendix figure** — same style as Figure 5e in cortical thickness, showing **both positive and
negative Hedges' g simultaneously** for cortical volume:

- **Light pink** (`lightpink`): g < 0 — first-named group has *less* cortical volume
- **Light blue** (`#AED6F1`): g > 0 — first-named group has *more* cortical volume
- **White**: no difference (g ≈ 0); **Grey85**: no data

All 68 parcels shown regardless of significance. Scale limits ±0.30.

In [ ]:
CNAMES_DISPLAY <- c(
  "De Novo PD vs HC"           = "de novo PD vs HC",
  "Prodromal PD vs HC"         = "prodromal PD vs HC",
  "De Novo PD vs Prodromal PD" = "de novo PD vs prodromal PD"
)

G_LIMIT_PASTEL <- 0.30

scale_diverging_pastel_vol <- scale_fill_gradient2(
  low      = "lightpink",
  mid      = "white",
  high     = "#AED6F1",
  midpoint = 0,
  limits   = c(-G_LIMIT_PASTEL, G_LIMIT_PASTEL),
  oob      = scales::squish,
  name     = "Hedges' g",
  na.value = "grey85"
)

if (!exists("dk_4view")) {
  dk_4view <- dk()
  dk_4view$data[[1]] <- dk_4view$data[[1]] %>% filter(view %in% c("lateral", "medial"))
}

make_pastel_vol <- function(cname, fdr_only = FALSE) {
  d <- df_stats %>% filter(contrast == cname)
  if (fdr_only) {
    plot_data <- d %>% mutate(g_plot = if_else(p_fdr < 0.05, g, NA_real_)) %>% select(label, g_plot)
  } else {
    plot_data <- d %>% mutate(g_plot = g) %>% select(label, g_plot)
  }
  ggplot(plot_data) +
    geom_brain(
      atlas    = dk_4view,
      mapping  = aes(fill = g_plot),
      colour   = "grey70",
      position = position_brain("horizontal")
    ) +
    scale_diverging_pastel_vol +
    labs(title = CNAMES_DISPLAY[[cname]]) +
    theme_void() +
    theme(
      plot.title    = element_text(hjust = 0.5, size = 11, face = "bold",
                                   margin = margin(t = 10, b = 3)),
      legend.position = "right",
      plot.margin   = margin(t = 8, r = 8, b = 12, l = 8)
    )
}

# Figure 5e-1: unthresholded (all 68 parcels)
options(repr.plot.width = 10, repr.plot.height = 12)

panels_unc <- map(names(contrasts), ~make_pastel_vol(.x, fdr_only = FALSE))

fig_diverging_unc <- wrap_plots(panels_unc, ncol = 1) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "Cortical Volume: Diverging Hedges' g (unthresholded)",
    subtitle = "Light pink = volume loss (g < 0) | Light blue = volume gain (g > 0) | Scale ±0.30",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

print(fig_diverging_unc)
ggsave(file.path(FIG_DIR, "appendix_diverging_volume_cortical_unthresh.png"),
       fig_diverging_unc, width = 10, height = 12, dpi = 300)
cat("Saved: appendix_diverging_volume_cortical_unthresh.png\n")

# Figure 5e-2: FDR-masked
panels_fdr <- map(names(contrasts), ~make_pastel_vol(.x, fdr_only = TRUE))

fig_diverging_fdr <- wrap_plots(panels_fdr, ncol = 1) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "Cortical Volume: Diverging Hedges' g (FDR q < 0.05)",
    subtitle = "Light pink = volume loss (g < 0) | Light blue = volume gain (g > 0) | Grey = not significant",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

print(fig_diverging_fdr)
ggsave(file.path(FIG_DIR, "appendix_diverging_volume_cortical_fdr.png"),
       fig_diverging_fdr, width = 10, height = 12, dpi = 300)
cat("Saved: appendix_diverging_volume_cortical_fdr.png\n")